# NB-07_salida_cg_imputa

Process Flow SAS: **Salida CG Imputa** — `ProcessFlowContainer-diHU2ZUCEv1lqT51`

In [ ]:
# ========= Parámetros =========
# Variables macro del SAS original. El .egp NO las define (venían del
# entorno SAS): su valor sale de la entrevista B4 o de
# project_config.yaml → run.macro_params, o se inyecta acá
# (celda 'parameters' de papermill).

Year = None  # &Year — nadie declaró su valor

faltantes = [n for n, v in {"Year": Year}.items() if v is None]
if faltantes:
    raise ValueError(f"Parámetros sin valor: {faltantes}")

In [ ]:
# ========= Celda 1: Configuración =========
import pandas as pd
import numpy as np
import os
import sqlalchemy
import datetime
from pathlib import Path
from sqlalchemy import text

# Conexión a BD — la define el orquestador al ejecutar
engine = sqlalchemy.create_engine(os.environ["SASMIG_DB_URL"], pool_pre_ping=True)
# Sesión de BD del notebook — espejo de la sesión WORK de SAS: las
# tablas temporales #tmp viven en ESTA conexión y mueren al cerrar el
# kernel. AUTOCOMMIT: cada statement commitea, como los pasos de SAS.
work_conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

# Logging liviano de resultados — aprobado en la entrevista (Fase 4)
_LOG_PATH = Path("log") / "NB-07_salida_cg_imputa.log"
_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
def _log(label, value=None):
    """Una línea por celda: imprime y persiste. Jamás rompe la corrida."""
    try:
        if hasattr(value, "shape"):
            detail = f"{value.shape[0]} filas x {value.shape[1]} cols"
        elif isinstance(value, int):
            detail = f"{value} filas"
        elif value is None:
            detail = "ok"
        else:
            detail = str(value)
        line = f"[{datetime.datetime.now():%Y-%m-%d %H:%M:%S}] {label}: {detail}"
        print(line)
        with open(_LOG_PATH, "a", encoding="utf-8") as fh:
            fh.write(line + "\n")
    except Exception:
        pass  # el log nunca puede tumbar el notebook
with open(_LOG_PATH, "a", encoding="utf-8") as _fh:
    _fh.write(f"\n=== corrida {datetime.datetime.now():%Y-%m-%d %H:%M:%S} ===\n")


## 1_Cobertura

Detecta entidades con información de ejecución presupuestaria incompleta en el año vigente respecto del mes máximo informado, marca los casos a revisar y reclasifica la partida 62 a 32 desde 2025

*confianza: medium · verificador: approve · SAS: PROC SQL con tablas WORK temporales (session_sql: #ejec_cgr, #cuenta_meses, #max_mes, #max_mes_ent, #cobertura_mes, #detalle_entidad, #cobertura_detalle) + PROC TRANSPOSE y PROC SORT NODUPKEY resueltos en pandas por depender del orden de filas*

In [ ]:
# ========= 1_Cobertura =========
# /*i. DETERMINA FALTANTES EN INFO CONTRALORIA*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cgr"))
sql_ejec_cgr = f"""
SELECT
    CONCAT_WS('_', AÑO, MES) AS PERIODO,
    MONEDA,
    AÑO,
    MES,
    C_ENTIDAD,
    CAST(C_ENTIDAD AS varchar(10)) AS ENTIDAD,
    C_PARTIDA,
    C_CAPITULO,
    C_PROGRAMA,
    CONCAT_WS('_', C_PARTIDA, C_CAPITULO, C_PROGRAMA) AS PCP,
    CONCAT_WS('_', N_PARTIDA, N_CAPITULO, N_PROGRAMA) AS N_PCP,
    SUM(DEVENG) / 1000.0 AS DATO,
    CASE WHEN AÑO = {int(Year)} THEN 'Pt' ELSE 'Pt_1' END AS TIEMPO
INTO #ejec_cgr
FROM GOBGENER.dbo.EJECUCION
WHERE AÑO = {int(Year)} OR (AÑO = {int(Year)} - 1 AND MES = 12)
GROUP BY AÑO, MES, MONEDA, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, N_PARTIDA, N_CAPITULO, N_PROGRAMA
"""
work_conn.execute(text(sql_ejec_cgr))


In [ ]:
# UPDATE EJEC_CGR SET DATO=. WHERE DATO IN (0,.,) -> nulifica ceros y valores ya ausentes
res = work_conn.execute(text("UPDATE #ejec_cgr SET DATO = NULL WHERE DATO = 0 OR DATO IS NULL"))
_log("UPDATE #ejec_cgr (DATO nulificado)", res.rowcount)


In [ ]:
# nota SAS: queda comentado max(t1.mes*(t1.Tiempo='Pt')) as m_mes -> código anulado, no se traduce
work_conn.execute(text("DROP TABLE IF EXISTS #cuenta_meses"))
sql_cuenta_meses = """
SELECT TIEMPO, C_ENTIDAD, MONEDA, COUNT(DATO) AS N
INTO #cuenta_meses
FROM #ejec_cgr
GROUP BY TIEMPO, C_ENTIDAD, MONEDA
"""
work_conn.execute(text(sql_cuenta_meses))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #max_mes"))
sql_max_mes = """
SELECT TIEMPO, MAX(MES) AS MAX_MES
INTO #max_mes
FROM #ejec_cgr
WHERE TIEMPO = 'Pt'
GROUP BY TIEMPO
"""
work_conn.execute(text(sql_max_mes))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #max_mes_ent"))
sql_max_mes_ent = """
SELECT TIEMPO, C_ENTIDAD, MONEDA, MAX(MES) AS MAX_MES
INTO #max_mes_ent
FROM #ejec_cgr
WHERE TIEMPO = 'Pt' AND DATO IS NOT NULL
GROUP BY TIEMPO, C_ENTIDAD, MONEDA
"""
work_conn.execute(text(sql_max_mes_ent))


In [ ]:
# PROC TRANSPOSE DATA=Cuenta_Meses OUT=TRANS_MES; BY C_ENTIDAD MONEDA; ID TIEMPO
# depende del orden de filas (BY-group) -> se resuelve en pandas, no en SQL
cuenta_meses = pd.read_sql(
    text("SELECT TIEMPO, C_ENTIDAD, MONEDA, N FROM #cuenta_meses ORDER BY C_ENTIDAD, MONEDA, TIEMPO"),
    work_conn,
)
trans_mes = cuenta_meses.pivot(index=["C_ENTIDAD", "MONEDA"], columns="TIEMPO", values="N").reset_index()
trans_mes.columns.name = None
_log("trans_mes", trans_mes)
trans_mes.to_sql("#trans_mes", work_conn, if_exists="replace", index=False)


In [ ]:
# /*Detecta cobertura faltante*/
# Pt ausente (NULL) equivale al missing de SAS: es "menor que todo", igual que Pt < max_mes
work_conn.execute(text("DROP TABLE IF EXISTS #cobertura_mes"))
sql_cobertura_mes = """
SELECT T1.C_ENTIDAD, T1.MONEDA, T1.Pt, T1.Pt_1, T2.MAX_MES,
    CASE WHEN T1.Pt IS NULL OR T1.Pt < T2.MAX_MES THEN 'Revisar' ELSE 'OK' END AS ID
INTO #cobertura_mes
FROM #trans_mes T1
CROSS JOIN #max_mes T2
"""
work_conn.execute(text(sql_cobertura_mes))


In [ ]:
# /*Define nombre de entidades y códigos claves*/
work_conn.execute(text("DROP TABLE IF EXISTS #detalle_entidad"))
sql_detalle_entidad = """
SELECT DISTINCT MONEDA, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, N_PCP
INTO #detalle_entidad
FROM #ejec_cgr
"""
work_conn.execute(text(sql_detalle_entidad))


In [ ]:
# PROC SORT DATA=DETALLE_ENTIDAD OUT=DETALLE_ENTIDAD_UN NODUPKEY BY MONEDA C_ENTIDAD
# depende del orden -> se resuelve en pandas
detalle_entidad = pd.read_sql(
    text("SELECT MONEDA, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, N_PCP FROM #detalle_entidad ORDER BY MONEDA, C_ENTIDAD"),
    work_conn,
)
detalle_entidad_un = detalle_entidad.drop_duplicates(subset=["MONEDA", "C_ENTIDAD"], keep="first")
_log("detalle_entidad_un", detalle_entidad_un)
detalle_entidad_un.to_sql("#detalle_entidad_un", work_conn, if_exists="replace", index=False)


In [ ]:
# /*FALTANTES CON DETALLE*/
# T3.MAX_MES ausente (LEFT JOIN sin match) equivale al missing de SAS ("menor que todo"):
# por eso T2.MAX_MES > T3.MAX_MES también debe cumplirse cuando T3.MAX_MES es NULL
work_conn.execute(text("DROP TABLE IF EXISTS #cobertura_detalle"))
sql_cobertura_detalle = f"""
SELECT DISTINCT
    T1.MONEDA,
    {int(Year)} AS AÑO,
    T1.C_ENTIDAD,
    T1.C_PARTIDA,
    T1.C_CAPITULO,
    T1.C_PROGRAMA,
    T1.N_PCP,
    T2.ID,
    T2.MAX_MES,
    T3.MAX_MES AS MAX_MES_ENT
INTO #cobertura_detalle
FROM #detalle_entidad_un T1
LEFT JOIN #cobertura_mes T2 ON T1.C_ENTIDAD = T2.C_ENTIDAD AND T1.MONEDA = T2.MONEDA
LEFT JOIN #max_mes_ent T3 ON T1.C_ENTIDAD = T3.C_ENTIDAD AND T1.MONEDA = T3.MONEDA
WHERE T2.ID = 'Revisar' AND ((T2.MAX_MES > T3.MAX_MES OR T3.MAX_MES IS NULL) OR T2.Pt IS NULL)
"""
work_conn.execute(text(sql_cobertura_detalle))


In [ ]:
# DATA COBERTURA_DETALLE; SET COBERTURA_DETALLE; C_PARTIDA_2=C_PARTIDA;
work_conn.execute(text("ALTER TABLE #cobertura_detalle ADD C_PARTIDA_2 VARCHAR(10)"))
res = work_conn.execute(text("UPDATE #cobertura_detalle SET C_PARTIDA_2 = C_PARTIDA"))
_log("UPDATE #cobertura_detalle (C_PARTIDA_2 inicial)", res.rowcount)


In [ ]:
# UPDATE COBERTURA_DETALLE SET C_PARTIDA_2='32' WHERE C_PARTIDA='62' AND AÑO>=2025
res = work_conn.execute(
    text("UPDATE #cobertura_detalle SET C_PARTIDA_2 = '32' WHERE C_PARTIDA = '62' AND AÑO >= 2025")
)
_log("UPDATE #cobertura_detalle (reclasifica partida 62->32)", res.rowcount)


In [ ]:
# PROC SQL; DROP TABLE EJEC_CGR,CUENTA_MESES,MAX_MES,MAX_MES_ENT,TRANS_MES,DETALLE_ENTIDAD;
# limpia las tablas de sesión intermedias; COBERTURA_MES, DETALLE_ENTIDAD_UN y COBERTURA_DETALLE
# permanecen en la sesión como entregables del nodo
for _t in ["#ejec_cgr", "#cuenta_meses", "#max_mes", "#max_mes_ent", "#trans_mes", "#detalle_entidad"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## 2a_TCons Mixtas_M_CGR_Faltante

Redistribuye la ejecución CGR del periodo faltante en transacciones mixtas (M) aplicando los ponderadores de estructura de muestra (EST_MIXTAS_M / EST_EXP_MIXTAS_M) y datos DIPRES del año anterior, generando la ejecución total por entidad, cuenta e ítem para el análisis desagregado y agregado de coyuntura de gobierno general

*confianza: low · verificador: approve · SAS: PROC SQL multi-step con LEFT/INNER JOINs, GROUP BY y redistribución por ponderadores sobre tablas de BD y tablas temporales de sesión (#), más DATA steps de concatenación y DROP de columnas*

In [ ]:
# ========= 2a_TCons Mixtas_M_CGR_Faltante =========
# /*Ejecucion cgr del periodo, para faltantes*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cgr_i"))
sql_ejec_cgr_i = """
SELECT T1.*, T3.MAX_MES, T3.MAX_MES_ENT
INTO #ejec_cgr_i
FROM GOBGENER.dbo.EJECUCION T1
LEFT JOIN #cobertura_detalle T3
    ON T1.AÑO = T3.AÑO - 1
   AND T1.C_ENTIDAD = T3.C_ENTIDAD
   AND T1.C_PARTIDA = T3.C_PARTIDA_2
   AND T1.C_CAPITULO = T3.C_CAPITULO
   AND T1.C_PROGRAMA = T3.C_PROGRAMA
   AND T1.MONEDA = T3.MONEDA
WHERE T3.ID = 'Revisar'
  AND T1.MES <= T3.MAX_MES
  AND T1.MES > T3.MAX_MES_ENT
  AND T3.C_PARTIDA IN ('88', '62', '31')
  AND T3.C_PARTIDA_2 IN ('88', '32', '31')
"""
res = work_conn.execute(text(sql_ejec_cgr_i))
_log("CREATE #ejec_cgr_i", res.rowcount)


In [ ]:
# /*Ejecucion cgr del periodo, para GORES mes dipres*/
# /*determina mes max info CGR*/
work_conn.execute(text("DROP TABLE IF EXISTS #mes_max_cgr"))
sql_mes_max_cgr = f"""
SELECT AÑO, MAX(MES) AS MAX_MES
INTO #mes_max_cgr
FROM GOBGENER.dbo.EJECUCION
WHERE AÑO = {int(Year)}
GROUP BY AÑO
"""
res = work_conn.execute(text(sql_mes_max_cgr))
_log("CREATE #mes_max_cgr", res.rowcount)


In [ ]:
# /*data cgr año anterior mes siguiente para imputar GORE a DIPRES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cgr_i_b"))
sql_ejec_cgr_i_b = """
SELECT T1.*
INTO #ejec_cgr_i_b
FROM GOBGENER.dbo.EJECUCION T1
CROSS JOIN #mes_max_cgr T2
INNER JOIN GOBGENER.dbo.CR18_APERTURA T3 ON T1.C_ENTIDAD = T3.ENTIDAD
WHERE T1.AÑO = T2.AÑO - 1
  AND T1.MES = T2.MAX_MES + 1
  AND T3.NIVEL = 'REGIONAL'
"""
res = work_conn.execute(text(sql_ejec_cgr_i_b))
_log("CREATE #ejec_cgr_i_b", res.rowcount)


In [ ]:
# /*datos para imputar tribunales, y gores faltantes desde CGR, y también para imputar Gores para mes dipres*/
# data EJEC_CGR_I; set EJEC_CGR_I EJEC_CGR_I_b; -> EJEC_CGR_I_b no trae MAX_MES/MAX_MES_ENT:
# SAS rellena esas columnas con missing al unir; se replica con NULL explícito
sql_append_cgr_i_b = "INSERT INTO #ejec_cgr_i SELECT *, NULL, NULL FROM #ejec_cgr_i_b"
res = work_conn.execute(text(sql_append_cgr_i_b))
_log("APPEND #ejec_cgr_i <- #ejec_cgr_i_b", res.rowcount)

# proc sql; drop table EJEC_CGR_I_b, MES_MAX_CGR;
for _t in ["#ejec_cgr_i_b", "#mes_max_cgr"]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))

# data EJEC_CGR_I (drop=...): columnas de apoyo que ya cumplieron su función
work_conn.execute(text(
    "ALTER TABLE #ejec_cgr_i DROP COLUMN "
    "N_ITEM, N_ASIGNACION, C_SUBASIGNACION, N_SUBASIGNACION, "
    "PPTO_INICIAL, PPTO_ACTUAL, PERCIB_PAG, POR_PERCIB_PAG, MAX_MES, MAX_MES_ENT"
))


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION DE MUESTRA DE TRANSACCIONES MIXTAS (M) SEGUN ESTRUCTURA, PARA EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m"))
sql_ejec_muestra_mixtas_m = """
SELECT
    T2.SUBSECTOR, T2.MONEDA, T2.MES, T2.AÑO, T1.ENTIDAD AS C_ENTIDAD,
    T2.C_PARTIDA, T2.N_PARTIDA, T2.C_CAPITULO, T2.N_CAPITULO,
    T2.C_PROGRAMA, T2.N_PROGRAMA, T2.C_TIPO_CUENTA, T2.C_CUENTA,
    T2.N_CUENTA, T2.C_ITEM, T2.C_ASIGNACION, T2.C_ANALITICO, T2.N_ANALITICO,
    T2.DEVENG,
    T1.C_SCN, T1.C_CAGENTE, T1.C_CUP,
    SUM(T2.DEVENG / 1000000.0 * T1.POND) AS DEVENGO
INTO #ejec_muestra_mixtas_m
FROM GOBGENER.dbo.EST_MIXTAS_M T1
LEFT JOIN #ejec_cgr_i T2
    ON T2.C_ENTIDAD = T1.ENTIDAD
   AND T2.C_ASIGNACION = T1.C_ASIGNACION
   AND T2.C_ANALITICO = T1.C_ANALITICO
   AND T2.C_CUENTA = T1.C_CUENTA
   AND T2.C_ITEM = T1.C_ITEM
   AND T2.MES = T1.MES
WHERE T2.C_CUENTA IN ('24', '33')
  AND T2.C_ITEM IN ('03', '01', '06', '08', '09')
  AND T1.ENTIDAD NOT IN (5601, 10201)
  AND T2.DEVENG IS NOT NULL
GROUP BY
    T2.SUBSECTOR, T2.MONEDA, T2.MES, T2.AÑO, T1.ENTIDAD,
    T2.C_PARTIDA, T2.N_PARTIDA, T2.C_CAPITULO, T2.N_CAPITULO,
    T2.C_PROGRAMA, T2.N_PROGRAMA, T2.C_TIPO_CUENTA, T2.C_CUENTA,
    T2.N_CUENTA, T2.C_ITEM, T2.C_ASIGNACION, T2.C_ANALITICO, T2.N_ANALITICO,
    T2.DEVENG, T1.C_SCN, T1.C_CAGENTE, T1.C_CUP
"""
res = work_conn.execute(text(sql_ejec_muestra_mixtas_m))
_log("CREATE #ejec_muestra_mixtas_m", res.rowcount)

# DATA EJEC_MUESTRA_MIXTAS_M (DROP=DEVENG): el bruto solo servía para no colapsar transacciones distintas en el GROUP BY
work_conn.execute(text("ALTER TABLE #ejec_muestra_mixtas_m DROP COLUMN DEVENG"))


In [ ]:
# /* CREA TABLA DE ANALITICOS DE MIXTAS (M) A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mixtas_m_2"))
sql_cruce_ejec_exp_mixtas_m_2 = """
SELECT
    T2.SUBSECTOR, T2.MONEDA, T2.MES, T2.AÑO, T2.C_ENTIDAD,
    T2.C_PARTIDA, T2.N_PARTIDA, T2.C_CAPITULO, T2.N_CAPITULO,
    T2.C_PROGRAMA, T2.N_PROGRAMA, T2.C_TIPO_CUENTA, T2.C_CUENTA,
    T2.N_CUENTA, T2.C_ITEM, T2.C_ASIGNACION, T2.C_ANALITICO, T2.N_ANALITICO,
    T2.DEVENG,
    T1.Clasif_Tcons
INTO #cruce_ejec_exp_mixtas_m_2
FROM GOBGENER.dbo.ANALIT_TCONS T1
LEFT JOIN #ejec_cgr_i T2
    ON T2.C_ENTIDAD = T1.ENTIDAD
   AND T2.C_ASIGNACION = T1.C_ASIGNACION
   AND T2.C_ANALITICO = T1.C_ANALITICO
   AND T2.C_CUENTA = T1.C_CUENTA
   AND T2.C_ITEM = T1.C_ITEM
WHERE T1.M_MUESTRA = 0
  AND T1.ENTIDAD NOT IN (5601, 10201)
  AND T1.Clasif_Tcons = 'M'
  AND T2.DEVENG IS NOT NULL
"""
res = work_conn.execute(text(sql_cruce_ejec_exp_mixtas_m_2))
_log("CREATE #cruce_ejec_exp_mixtas_m_2", res.rowcount)


In [ ]:
# /* CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MIXTAS (M), PARA EJECUCION AÑO t */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim2"))
sql_ejec_exp_mixtas_m_prelim2 = """
SELECT
    T2.SUBSECTOR, T2.MONEDA, T2.AÑO, T2.MES, T2.C_ENTIDAD,
    T2.C_PARTIDA, T2.N_PARTIDA, T2.C_CAPITULO, T2.N_CAPITULO,
    T2.C_PROGRAMA, T2.N_PROGRAMA, T2.C_TIPO_CUENTA, T2.C_CUENTA,
    T2.N_CUENTA, T2.C_ITEM, T2.C_ASIGNACION, T2.C_ANALITICO, T2.N_ANALITICO,
    T2.DEVENG, T2.Clasif_Tcons,
    T1.C_SCN, T1.C_CAGENTE, T1.C_CUP,
    SUM(T2.DEVENG) AS DATO_OR,
    SUM(
        CASE WHEN T3.DATO IS NOT NULL
             THEN T2.DEVENG / 1000000.0 * T1.POND * T3.DATO
             ELSE T2.DEVENG / 1000000.0 * T1.POND
        END
    ) AS DEVENGO
INTO #ejec_exp_mixtas_m_prelim2
FROM GOBGENER.dbo.EST_EXP_MIXTAS_M T1
LEFT JOIN #cruce_ejec_exp_mixtas_m_2 T2
    ON T2.C_CUENTA = T1.C_CUENTA
   AND T2.C_ITEM = T1.C_ITEM
   AND T2.C_ASIGNACION = T1.C_ASIGNACION
   AND T2.C_ANALITICO = T1.C_ANALITICO
   AND T2.MES = T1.MES
   AND T2.C_ENTIDAD = T1.ENTIDAD
LEFT JOIN #prom_mes T3
    ON T2.MONEDA = T3.MONEDA
   AND T2.MES = T3.MES
   AND T2.AÑO = T3.AÑO
WHERE T2.DEVENG IS NOT NULL
GROUP BY
    T2.SUBSECTOR, T2.MONEDA, T2.AÑO, T2.MES, T2.C_ENTIDAD,
    T2.C_PARTIDA, T2.N_PARTIDA, T2.C_CAPITULO, T2.N_CAPITULO,
    T2.C_PROGRAMA, T2.N_PROGRAMA, T2.C_TIPO_CUENTA, T2.C_CUENTA,
    T2.N_CUENTA, T2.C_ITEM, T2.C_ASIGNACION, T2.C_ANALITICO, T2.N_ANALITICO,
    T2.DEVENG, T2.Clasif_Tcons, T1.C_SCN, T1.C_CAGENTE, T1.C_CUP
"""
res = work_conn.execute(text(sql_ejec_exp_mixtas_m_prelim2))
_log("CREATE #ejec_exp_mixtas_m_prelim2", res.rowcount)


In [ ]:
# DATA EJEC_EXP_MIXTAS_M_prelim (DROP=DEVENG DATO_OR Clasif_Tcons): descarta columnas de apoyo del cálculo
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim"))
work_conn.execute(text(
    "SELECT * INTO #ejec_exp_mixtas_m_prelim FROM #ejec_exp_mixtas_m_prelim2"
))
work_conn.execute(text(
    "ALTER TABLE #ejec_exp_mixtas_m_prelim DROP COLUMN DEVENG, DATO_OR, Clasif_Tcons"
))


In [ ]:
# /* Crea tabla con la ejecución total de transacciones Mixtas (M), Muestra + Resto */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mixtas_m_total"))
_cols_mixtas_m_total = (
    "SUBSECTOR, MONEDA, MES, AÑO, C_ENTIDAD, C_PARTIDA, N_PARTIDA, C_CAPITULO, N_CAPITULO, "
    "C_PROGRAMA, N_PROGRAMA, C_TIPO_CUENTA, C_CUENTA, N_CUENTA, C_ITEM, C_ASIGNACION, "
    "C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_CUP, DEVENGO"
)
sql_ejec_mixtas_m_total = f"""
SELECT {_cols_mixtas_m_total}
INTO #ejec_mixtas_m_total
FROM #ejec_muestra_mixtas_m
UNION ALL
SELECT {_cols_mixtas_m_total}
FROM #ejec_exp_mixtas_m_prelim
"""
res = work_conn.execute(text(sql_ejec_mixtas_m_total))
_log("CREATE #ejec_mixtas_m_total", res.rowcount)


In [ ]:
# /* Ajuste por imputación de D63 en pandemia (2020) */
res = work_conn.execute(text(
    "DELETE FROM #ejec_mixtas_m_total "
    "WHERE C_ENTIDAD = 911 AND C_CUENTA = '24' AND C_ITEM = '03' AND C_ASIGNACION = '002' "
    "AND C_ANALITICO IN ('03002000') AND AÑO = 2020 AND MES > 4"
))
_log("DELETE #ejec_mixtas_m_total (ajuste D63 pandemia)", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para ANALISIS DESAGREGADO CG coyuntura */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt_aper"))
sql_ejec_g_mixt_aper = f"""
SELECT
    T1.SUBSECTOR,
    T1.MONEDA,
    {int(Year)} AS AÑO,
    CASE WHEN T1.MES IN (1, 2, 3) THEN 1
         WHEN T1.MES IN (4, 5, 6) THEN 2
         WHEN T1.MES IN (7, 8, 9) THEN 3
         ELSE 4 END AS TRIM,
    T1.MES,
    T1.C_ENTIDAD,
    T1.C_PARTIDA,
    T1.C_CAPITULO,
    T1.C_PROGRAMA,
    T1.C_TIPO_CUENTA,
    T1.C_CUENTA,
    T1.C_ITEM,
    T1.C_ASIGNACION,
    T1.C_ANALITICO,
    T1.N_ANALITICO,
    T1.C_SCN,
    T1.C_CAGENTE,
    SUM(T1.DEVENGO) AS Dato,
    T3.NIVEL,
    CASE WHEN T3.ACTIVIDAD = 'EEPP' THEN 'EEPP'
         WHEN T3.ACTIVIDAD = 'SSPP' THEN 'SSPP'
         ELSE 'AAPP' END AS C_ACT,
    'TC' AS C_ACTIVO
INTO #ejec_g_mixt_aper
FROM #ejec_mixtas_m_total T1
LEFT JOIN GOBGENER.dbo.CR18_APERTURA T3 ON T1.C_ENTIDAD = T3.ENTIDAD
GROUP BY
    T1.SUBSECTOR, T1.MONEDA, T1.AÑO,
    CASE WHEN T1.MES IN (1, 2, 3) THEN 1
         WHEN T1.MES IN (4, 5, 6) THEN 2
         WHEN T1.MES IN (7, 8, 9) THEN 3
         ELSE 4 END,
    T1.MES, T1.C_ENTIDAD, T1.C_PARTIDA, T1.C_CAPITULO, T1.C_PROGRAMA, T1.C_TIPO_CUENTA,
    T1.C_CUENTA, T1.C_ITEM, T1.C_ASIGNACION, T1.C_ANALITICO, T1.N_ANALITICO,
    T1.C_SCN, T1.C_CAGENTE, T3.NIVEL,
    CASE WHEN T3.ACTIVIDAD = 'EEPP' THEN 'EEPP'
         WHEN T3.ACTIVIDAD = 'SSPP' THEN 'SSPP'
         ELSE 'AAPP' END,
    'TC'
"""
res = work_conn.execute(text(sql_ejec_g_mixt_aper))
_log("CREATE #ejec_g_mixt_aper", res.rowcount)


In [ ]:
# /* SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para CG coyuntura, agregado */
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt"))
sql_ejec_g_mixt = """
SELECT
    T1.SUBSECTOR, T1.MONEDA, T1.AÑO, T1.TRIM, T1.MES, T1.C_TIPO_CUENTA, T1.C_CUENTA, T1.C_ITEM,
    T1.C_SCN, T1.C_CAGENTE, T1.C_ACTIVO,
    SUM(T1.Dato) AS Dato,
    T1.C_ACT, T1.NIVEL
INTO #ejec_g_mixt
FROM #ejec_g_mixt_aper T1
GROUP BY
    T1.SUBSECTOR, T1.MONEDA, T1.AÑO, T1.TRIM, T1.MES, T1.C_TIPO_CUENTA, T1.C_CUENTA, T1.C_ITEM,
    T1.C_SCN, T1.C_CAGENTE, T1.C_ACT, T1.NIVEL, T1.C_ACTIVO
"""
res = work_conn.execute(text(sql_ejec_g_mixt))
_log("CREATE #ejec_g_mixt", res.rowcount)


In [ ]:
# PROC SQL; DROP TABLE EJEC_MUESTRA_MIXTAS_M,CRUCE_EJEC_EXP_MIXTAS_M,EJEC_EXP_MIXTAS_M_prelim,
# EJEC_EXP_MIXTAS_M,EJEC_MIXTAS_M_TOTAL; QUIT;
# limpieza final de intermedias (incluye dos nombres del SAS original sin el sufijo con el
# que se crearon en este nodo -_2/_prelim2 faltante, probable typo-; IF EXISTS lo tolera
# igual que el DROP de SAS sobre una tabla inexistente, sin efecto real)
for _t in [
    "#ejec_muestra_mixtas_m",
    "#cruce_ejec_exp_mixtas_m",
    "#ejec_exp_mixtas_m_prelim",
    "#ejec_exp_mixtas_m",
    "#ejec_mixtas_m_total",
]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## 2b_Salida EJEC hasta CGR

Mapea la ejecución presupuestaria de Contraloría a conceptos SCN (D1, D63, P2, etc.) por entidad y periodo, aplica el ajuste de transferencias en especie y produce las salidas detallada y agregada de imputaciones CGR

*confianza: medium · verificador: approve · SAS: Secuencia PROC SQL sobre WORK materializados como #tmp (work_conn): joins contra tablas de estructura SCN, ajuste CASE/UPDATE de transferencias en especie, UNION ALL y agregaciones GROUP BY finales*

In [ ]:
# ========= 2b_Salida EJEC hasta CGR =========
# /*Analiticos predefinidos */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim"))
sql_ejecucion_a_prelim = """
SELECT
    t1.SUBSECTOR,
    t1.MONEDA,
    t1.AÑO,
    CASE WHEN t1.MES IN (1,2,3) THEN 1
         WHEN t1.MES IN (4,5,6) THEN 2
         WHEN t1.MES IN (7,8,9) THEN 3
         ELSE 4 END AS TRIM,
    t1.MES,
    t1.C_ENTIDAD,
    t1.C_PARTIDA,
    t1.C_CAPITULO,
    t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA,
    t1.C_CUENTA,
    t1.C_ITEM,
    t1.C_ASIGNACION,
    t1.C_ANALITICO,
    t1.N_ANALITICO,
    CASE WHEN t2.ANALITICO = 'M' THEN t1.N_ANALITICO ELSE t2.ANALITICO END AS ANALITICO_,
    t2.C_SCN,
    t2.C_CAGENTE,
    t2.C_TRANSACCION,
    t2.T_TRANSACCION,
    t2.N_CUENTA,
    t2.C_ACTIVO,
    t2.C_CUP,
    CASE WHEN t4.ACTIVIDAD = 'FFAA' THEN 'AAPP'
         WHEN t4.ACTIVIDAD = 'FFOO' THEN 'AAPP'
         WHEN t4.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
         WHEN t4.ACTIVIDAD = 'FONASA' THEN 'AAPP'
         WHEN t4.ACTIVIDAD = 'CORFO' THEN 'AAPP'
         WHEN t4.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
         WHEN t4.ACTIVIDAD = 'AAPP' THEN 'AAPP'
         WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP'
         WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
    t4.NIVEL,
    (t1.DEVENG * t3.DATO / 1000000) AS DEVENGO
INTO #ejecucion_a_prelim
FROM #ejec_cgr_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA NOT IN ('05','13','24','33')
  AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.C_ENTIDAD NOT IN (5601,10201)
"""
work_conn.execute(text(sql_ejecucion_a_prelim))

In [ ]:
# /*Este Ajuste lo pidieron de síntesis, incorpora al indicador de transferencias en esp lo que consideran desde anuales, el porcentaje sale de anuales...*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_aj"))
sql_ejecucion_a_prelim_aj = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.ANALITICO_,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
    CASE WHEN t1.C_CUENTA = '22' AND SUBSTRING(t1.C_ANALITICO,1,5) = '12999' AND t1.C_PARTIDA = '16' AND t1.C_ACT = 'SSPP'
         THEN t1.DEVENGO * 0.928 ELSE t1.DEVENGO END AS DEVENGO
INTO #ejecucion_a_prelim_aj
FROM #ejecucion_a_prelim t1
"""
work_conn.execute(text(sql_ejecucion_a_prelim_aj))

In [ ]:
# UPDATE EJECUCION_A_prelim_Aj: reclasifica a D63/transfesp la porción de transferencias en especie
sql_update_aj = """
UPDATE #ejecucion_a_prelim_aj
SET C_SCN = 'D63', C_CAGENTE = 'S14', C_TRANSACCION = '433', T_TRANSACCION = 'transfesp', N_CUENTA = 'IG', C_CUP = '272'
WHERE C_CUENTA = '22' AND SUBSTRING(C_ANALITICO,1,5) = '12999' AND C_PARTIDA = '16' AND C_ACT = 'SSPP'
"""
res = work_conn.execute(text(sql_update_aj))
_log("UPDATE #ejecucion_a_prelim_aj", res.rowcount)

In [ ]:
# Recalcula EJECUCION_A_prelim con el complemento (1-0.928) para la misma porción ajustada en el paso anterior
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_v2"))
sql_ejecucion_a_prelim_v2 = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.ANALITICO_,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
    CASE WHEN t1.C_CUENTA = '22' AND SUBSTRING(t1.C_ANALITICO,1,5) = '12999' AND t1.C_PARTIDA = '16' AND t1.C_ACT = 'SSPP'
         THEN t1.DEVENGO * (1 - 0.928) ELSE t1.DEVENGO END AS DEVENGO
INTO #ejecucion_a_prelim_v2
FROM #ejecucion_a_prelim t1
"""
work_conn.execute(text(sql_ejecucion_a_prelim_v2))

In [ ]:
# Subconjunto de EJECUCION_A_prelim_Aj ya reclasificado (D63), para anexarlo (FORCE) a la versión complementaria
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_aj_filtered"))
sql_ejecucion_a_prelim_aj_filtered = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.ANALITICO_,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
    t1.DEVENGO
INTO #ejecucion_a_prelim_aj_filtered
FROM #ejecucion_a_prelim_aj t1
WHERE t1.C_CUENTA = '22' AND SUBSTRING(t1.C_ANALITICO,1,5) = '12999' AND t1.C_PARTIDA = '16' AND t1.C_ACT = 'SSPP'
"""
work_conn.execute(text(sql_ejecucion_a_prelim_aj_filtered))

In [ ]:
# PROC DATASETS APPEND BASE=EJECUCION_A_prelim DATA=EJECUCION_A_prelim_Aj FORCE
cols_ejecucion_a_prelim = ("SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, "
    "C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, ANALITICO_, "
    "C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, DEVENGO")
sql_append_prelim = f"INSERT INTO #ejecucion_a_prelim_v2 ({cols_ejecucion_a_prelim}) SELECT {cols_ejecucion_a_prelim} FROM #ejecucion_a_prelim_aj_filtered"
res = work_conn.execute(text(sql_append_prelim))
_log("APPEND #ejecucion_a_prelim_v2", res.rowcount)

In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_v2 WHERE C_CUENTA = '32' AND C_ANALITICO IN ('04001000','04002000')"))
_log("DELETE #ejecucion_a_prelim_v2 (04001000/04002000)", res.rowcount)

In [ ]:
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
#   que se desagrega anualmente entre ventas y D45 */
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_v2 WHERE C_ENTIDAD = 3801 AND C_CUENTA = '06' AND C_ITEM = '01' "
    "AND C_ASIGNACION = '000' AND C_ANALITICO IN ('01000000')"
))
_log("DELETE #ejecucion_a_prelim_v2 (CORFO)", res.rowcount)

In [ ]:
# /*Analiticos predefinidos (sigue)*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a"))
sql_ejecucion_a = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
    SUM(t1.DEVENGO) AS Dato
INTO #ejecucion_a
FROM #ejecucion_a_prelim_v2 t1
WHERE t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL
"""
work_conn.execute(text(sql_ejecucion_a))

In [ ]:
# /*Analiticos variables */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b"))
sql_ejecucion_b = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS TRIM,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    CASE WHEN t4.ACTIVIDAD = 'FFAA' THEN 'AAPP' WHEN t4.ACTIVIDAD = 'FFOO' THEN 'AAPP' WHEN t4.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
         WHEN t4.ACTIVIDAD = 'FONASA' THEN 'AAPP' WHEN t4.ACTIVIDAD = 'CORFO' THEN 'AAPP' WHEN t4.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
         WHEN t4.ACTIVIDAD = 'AAPP' THEN 'AAPP' WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP' WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
    t4.NIVEL,
    CASE WHEN t3.DATO IS NOT NULL THEN t1.DEVENG * t3.DATO / 1000000 ELSE t1.DEVENG / 1000000 END AS Dato
INTO #ejecucion_b
FROM #ejec_cgr_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
       AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA = '32' AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.C_ANALITICO IN ('04001000','04002000') AND t1.C_ENTIDAD NOT IN (5601,10201)
"""
work_conn.execute(text(sql_ejecucion_b))

In [ ]:
# /*analíticos variables*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c"))
sql_ejecucion_c = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS TRIM,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    CASE WHEN t4.ACTIVIDAD = 'FFAA' THEN 'AAPP' WHEN t4.ACTIVIDAD = 'FFOO' THEN 'AAPP' WHEN t4.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
         WHEN t4.ACTIVIDAD = 'FONASA' THEN 'AAPP' WHEN t4.ACTIVIDAD = 'CORFO' THEN 'AAPP' WHEN t4.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
         WHEN t4.ACTIVIDAD = 'AAPP' THEN 'AAPP' WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP' WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
    t4.NIVEL,
    (t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_c
FROM #ejec_cgr_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
       AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
       AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t1.C_CUENTA IN ('05','13','24','33') AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_SCN NOT IN ('TC')
"""
work_conn.execute(text(sql_ejecucion_c))

In [ ]:
# /* Transacciones diferenciadas para coyuntura */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d"))
sql_ejecucion_d = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS TRIM,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_ACTIVO AS C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    CASE WHEN t4.ACTIVIDAD = 'FFAA' THEN 'AAPP' WHEN t4.ACTIVIDAD = 'FFOO' THEN 'AAPP' WHEN t4.ACTIVIDAD = 'PREVISION' THEN 'AAPP'
         WHEN t4.ACTIVIDAD = 'FONASA' THEN 'AAPP' WHEN t4.ACTIVIDAD = 'CORFO' THEN 'AAPP' WHEN t4.ACTIVIDAD = 'CENABAST' THEN 'AAPP'
         WHEN t4.ACTIVIDAD = 'AAPP' THEN 'AAPP' WHEN t4.ACTIVIDAD = 'SSPP' THEN 'SSPP' WHEN t4.ACTIVIDAD = 'EEPP' THEN 'EEPP' END AS C_ACT,
    t4.NIVEL,
    (t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_d
FROM #ejec_cgr_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
       AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
       AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t2.OBS IN ('CR18_difcoy') AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND t2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11') AND t2.C_SCN NOT IN ('TC')
"""
work_conn.execute(text(sql_ejecucion_d))

In [ ]:
# /* Transacciones diferenciadas por actividad */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f"))
sql_ejecucion_f = """
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS TRIM,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    t2.C_CUP AS C_ACT, t4.NIVEL,
    (t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_f
FROM #ejec_cgr_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
       AND t1.C_ENTIDAD = t2.ENTIDAD AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA
       AND t1.C_ITEM = t2.C_ITEM AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
LEFT JOIN GOBGENER.dbo.CR18_APERTURA t4
    ON t1.C_ENTIDAD = t4.ENTIDAD
WHERE t2.OBS IN ('CR18_difact') AND t1.C_ENTIDAD NOT IN (5601,10201) AND t2.C_SCN NOT IN ('TC')
  AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
work_conn.execute(text(sql_ejecucion_f))

In [ ]:
# /* Transacciones TCONS (sólo aplica Mixtas) EJEC_G_MIXT_aper*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g"))
sql_ejecucion_g = """
SELECT * INTO #ejecucion_g
FROM #ejec_g_mixt_aper
WHERE C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
"""
work_conn.execute(text(sql_ejecucion_g))

In [ ]:
# /*UNE TABLAS de Salida total de ejecución CG coy de data CGR*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_cgr_prev_2"))
cols_union = ("SUBSECTOR, MONEDA, AÑO, TRIM, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA, "
    "C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_TRANSACCION, T_TRANSACCION, "
    "N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato")
sql_union = f"""
SELECT {cols_union} INTO #ejecucion_cg_cgr_prev_2 FROM #ejecucion_a
UNION ALL SELECT {cols_union} FROM #ejecucion_b
UNION ALL SELECT {cols_union} FROM #ejecucion_c
UNION ALL SELECT {cols_union} FROM #ejecucion_d
UNION ALL SELECT {cols_union} FROM #ejecucion_g
UNION ALL SELECT {cols_union} FROM #ejecucion_f
"""
work_conn.execute(text(sql_union))

In [ ]:
# PROC SQL; DROP TABLE EJECUCION_A_prelim, EJECUCION_A, EJECUCION_B, EJECUCION_C, EJECUCION_D, EJECUCION_G, EJECUCION_F, EJECUCION_A_PRELIM_AJ
for _t in [
    "#ejecucion_a_prelim",
    "#ejecucion_a_prelim_v2",
    "#ejecucion_a",
    "#ejecucion_b",
    "#ejecucion_c",
    "#ejecucion_d",
    "#ejecucion_g",
    "#ejecucion_f",
    "#ejecucion_a_prelim_aj",
    "#ejecucion_a_prelim_aj_filtered",
]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))

In [ ]:
# /*SALIDA DETALLADA CGR - IMPUTACIONES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_cgr_aper_imp"))
sql_aper_imp = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, {int(Year)} AS AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
    SUM(t1.DATO) AS DATO, 'Imputación cobertura' AS OBS
INTO #ejecucion_cg_cgr_aper_imp
FROM #ejecucion_cg_cgr_prev_2 t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL
"""
res = work_conn.execute(text(sql_aper_imp))
_log("ejecucion_cg_cgr_aper_imp", res.rowcount)

In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_cg_cgr_aper_imp WHERE DATO = 0"))
_log("DELETE #ejecucion_cg_cgr_aper_imp (DATO=0)", res.rowcount)

In [ ]:
# /*SALIDA AGREGADA CGR - IMPUTACIONES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_cgr_imp"))
sql_imp = f"""
SELECT t1.SUBSECTOR, t1.MONEDA, {int(Year)} AS AÑO, t1.TRIM, t1.MES,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
    SUM(t1.DATO) AS DATO, 'Imputación cobertura' AS OBS
INTO #ejecucion_cg_cgr_imp
FROM #ejecucion_cg_cgr_prev_2 t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.TRIM, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL
"""
res = work_conn.execute(text(sql_imp))
_log("ejecucion_cg_cgr_imp", res.rowcount)

In [ ]:
res = work_conn.execute(text("DELETE FROM #ejecucion_cg_cgr_imp WHERE DATO = 0"))
_log("DELETE #ejecucion_cg_cgr_imp (DATO=0)", res.rowcount)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_cgr_prev_2"))

## 3a_Imputa_Mixtas_Faltantes

Imputa entidad, actividad y nivel faltantes en la ejecución DIPRES del año usando la información del año anterior y la cobertura CGR, valoriza la muestra de transacciones Mixtas (M) y expande el resto según la estructura muestral, generando la salida agregada y desagregada para el cierre de Gobierno Central

*confianza: low · verificador: approve · SAS: PROC SQL: múltiples CREATE TABLE con LEFT JOIN condicionales (CASE para imputar entidad/actividad/nivel), agregación de grupo completo vía ROW_NUMBER()+SUM() OVER, UNION ALL equivalente al DATA step SET, y limpieza final de tablas de trabajo*

In [ ]:
# ========= 3a_Imputa_Mixtas_Faltantes =========
# II. IMPUTA FALTANTES EN INFO CONTRALORIA USANDO DATOS DIPRES
work_conn.execute(text("DROP TABLE IF EXISTS #entid_dip_y1"))
sql_entid_dip_y1 = f"""
SELECT DISTINCT *
INTO #entid_dip_y1
FROM GOBGENER.dbo.CR18_ENTID_DIP_Y
WHERE AÑO = {int(Year) - 1}
"""
res = work_conn.execute(text(sql_entid_dip_y1))
_log("CREATE #entid_dip_y1", res.rowcount)


In [ ]:
# TRABAJA TCONS MIXTAS DIPRES
# Ejecucion dipres del periodo, con entidad asignada
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_dip_i"))
sql_ejec_dip_i = f"""
SELECT
    T1.*,
    CASE
        WHEN T2.C_ENTIDAD IS NULL AND T4.C_ENTIDAD IS NOT NULL THEN T4.C_ENTIDAD
        WHEN T4.C_ENTIDAD IS NULL AND T2.C_ENTIDAD IS NULL
            THEN TRY_CAST(CONCAT(LTRIM(RTRIM(T1.C_PARTIDA)), LTRIM(RTRIM(T1.C_CAPITULO)), LTRIM(RTRIM(T1.C_PROGRAMA))) AS INT)
        ELSE T2.C_ENTIDAD
    END AS C_ENTIDAD,
    CASE
        WHEN (T2.ACTIVIDAD IS NULL OR T2.ACTIVIDAD = '') AND (T4.ACTIVIDAD IS NOT NULL AND T4.ACTIVIDAD <> '')
            THEN T4.ACTIVIDAD
        ELSE T2.ACTIVIDAD
    END AS ACTIVIDAD,
    CASE
        WHEN (T2.NIVEL IS NULL OR T2.NIVEL = '') AND (T4.NIVEL IS NOT NULL AND T4.NIVEL <> '')
            THEN T4.NIVEL
        ELSE T2.NIVEL
    END AS NIVEL,
    T3.MAX_MES, T3.MAX_MES_ENT
INTO #ejec_dip_i
FROM GOBGENER.dbo.EJEC_DIPRES T1
LEFT JOIN GOBGENER.dbo.CR18_ENTID_DIP_Y T2
    ON T1.AÑO = T2.AÑO AND T1.C_PARTIDA = T2.C_PARTIDA AND T1.C_CAPITULO = T2.C_CAPITULO AND T1.C_PROGRAMA = T2.C_PROGRAMA
LEFT JOIN #entid_dip_y1 T4
    ON T1.C_PARTIDA = T4.C_PARTIDA AND T1.C_CAPITULO = T4.C_CAPITULO AND T1.C_PROGRAMA = T4.C_PROGRAMA
LEFT JOIN #cobertura_detalle T3
    ON T1.AÑO = T3.AÑO AND T1.C_PARTIDA = T3.C_PARTIDA AND T1.C_CAPITULO = T3.C_CAPITULO AND T1.C_PROGRAMA = T3.C_PROGRAMA AND T1.MONEDA = T3.MONEDA
WHERE T1.AÑO = {int(Year)} AND T3.ID = 'Revisar' AND T1.MES <= T3.MAX_MES AND T1.MES > T3.MAX_MES_ENT
      AND T1.C_PARTIDA NOT IN ('88', '62', '31')
"""
res = work_conn.execute(text(sql_ejec_dip_i))
# DATA step DROP=: descarta columnas de presupuesto y de cobertura ya usadas
work_conn.execute(text(
    "ALTER TABLE #ejec_dip_i DROP COLUMN PPTO_INICIAL, PPTO_ACTUAL, EJEC_ACUM, MAX_MES, MAX_MES_ENT"
))
_log("CREATE #ejec_dip_i", res.rowcount)


In [ ]:
# TCONS MIXTAS DE LA MUESTRA DIPRES
# El GROUP BY de SAS cubre absolutamente todas las columnas no agregadas de t2.*;
# se replica con ROW_NUMBER()+SUM() OVER(PARTITION BY ...) en vez de enumerar
# el esquema completo heredado de #ejec_dip_i
work_conn.execute(text("DROP TABLE IF EXISTS #muestra_mixtas_m_prev_dip"))
_partition_muestra = """
    PARTITION BY t2.SUBSECTOR, t2.MONEDA, t2.MES, t2.AÑO, t2.C_PARTIDA, t2.C_CAPITULO, t2.C_PROGRAMA,
                 t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
                 t2.C_ENTIDAD, t2.ACTIVIDAD, t2.NIVEL, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP, t2.DEVENG
"""
sql_muestra_mixtas_m_prev_dip = f"""
WITH base AS (
    SELECT
        t2.*,
        t1.C_SCN,
        t1.C_CAGENTE,
        CAST(t1.C_CUP AS CHAR(4)) AS C_CUP,
        SUM(t2.DEVENG / 1000000.0 * t1.POND) OVER ({_partition_muestra}) AS DEVENGO,
        ROW_NUMBER() OVER ({_partition_muestra} ORDER BY (SELECT NULL)) AS _rn
    FROM GOBGENER.dbo.EST_MIXTAS_M t1
    LEFT JOIN #ejec_dip_i t2
        ON t2.C_ENTIDAD = t1.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION AND t2.C_ANALITICO = t1.C_ANALITICO
       AND t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM AND t2.MES = t1.MES
    WHERE t2.C_CUENTA IN ('24', '33') AND t2.C_ITEM IN ('03', '01', '06', '08', '09') AND t1.ENTIDAD NOT IN (5601, 10201)
)
SELECT * INTO #muestra_mixtas_m_prev_dip FROM base WHERE _rn = 1
"""
work_conn.execute(text(sql_muestra_mixtas_m_prev_dip))
work_conn.execute(text("ALTER TABLE #muestra_mixtas_m_prev_dip DROP COLUMN _rn"))

# EJEC_MUESTRA_MIXTAS_M_DIP_I: igual que la muestra previa, sin la columna DEVENG original
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_muestra_mixtas_m_dip_i"))
res = work_conn.execute(text(
    "SELECT * INTO #ejec_muestra_mixtas_m_dip_i FROM #muestra_mixtas_m_prev_dip"
))
work_conn.execute(text("ALTER TABLE #ejec_muestra_mixtas_m_dip_i DROP COLUMN DEVENG"))
work_conn.execute(text("DROP TABLE IF EXISTS #muestra_mixtas_m_prev_dip"))
_log("CREATE #ejec_muestra_mixtas_m_dip_i", res.rowcount)


In [ ]:
# CREA TABLA DE ANALITICOS DE MIXTAS (M) A EXPANDIR (FUERA DE MUESTRA), SEGUN EJECUCION AÑO t
work_conn.execute(text("DROP TABLE IF EXISTS #cruce_ejec_exp_mixtas_m_dip"))
# t1.C_CUENTA/C_ITEM/C_ASIGNACION/C_ANALITICO se omiten del SELECT (duplicarian a los de t2);
# el WHERE t2.DEVENG IS NOT NULL ya exige coincidencia con t1, por lo que los valores son iguales
sql_cruce_ejec_exp_mixtas_m_dip = """
SELECT
    t2.*,
    t1.ENTIDAD,
    t1.Clasif_Tcons
INTO #cruce_ejec_exp_mixtas_m_dip
FROM GOBGENER.dbo.ANALIT_TCONS t1
LEFT JOIN #ejec_dip_i t2
    ON t2.C_ENTIDAD = t1.ENTIDAD AND t2.C_ASIGNACION = t1.C_ASIGNACION AND t2.C_ANALITICO = t1.C_ANALITICO
   AND t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM
WHERE t1.M_MUESTRA = 0 AND t1.ENTIDAD NOT IN (5601, 10201) AND t1.Clasif_Tcons = 'M' AND t2.DEVENG IS NOT NULL
"""
res = work_conn.execute(text(sql_cruce_ejec_exp_mixtas_m_dip))
_log("CREATE #cruce_ejec_exp_mixtas_m_dip", res.rowcount)


In [ ]:
# CREA TABLA DE REDISTRIBUCION EXPANDIDA SEGUN ESTRUCTURA DE MUESTRA, PARA RESTO DE TRANSACCIONES MIXTAS (M), PARA EJECUCION AÑO t
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim_dip"))
_partition_exp = """
    PARTITION BY t2.SUBSECTOR, t2.MONEDA, t2.AÑO, t2.MES, t2.C_PARTIDA, t2.C_CAPITULO, t2.C_PROGRAMA,
                 t2.C_TIPO_CUENTA, t2.C_CUENTA, t2.C_ITEM, t2.C_ASIGNACION, t2.C_ANALITICO, t2.N_ANALITICO,
                 t2.C_ENTIDAD, t2.ACTIVIDAD, t2.NIVEL, t2.ENTIDAD, t2.Clasif_Tcons, t1.C_SCN, t1.C_CAGENTE, t1.C_CUP, t2.DEVENG
"""
# Orden C_SCN, C_CAGENTE, C_CUP alineado con EJEC_MUESTRA_MIXTAS_M_DIP_I para que el UNION ALL
# posterior (equivalente al SET por nombre de SAS) no desalinee columnas por posicion
sql_ejec_exp_mixtas_m_prelim_dip = f"""
WITH base AS (
    SELECT
        t2.*,
        t1.C_SCN,
        t1.C_CAGENTE,
        t1.C_CUP,
        SUM(t2.DEVENG) OVER ({_partition_exp}) AS DATO_OR,
        SUM(t2.DEVENG / 1000000.0 * t1.POND * t3.DATO) OVER ({_partition_exp}) AS DEVENGO,
        ROW_NUMBER() OVER ({_partition_exp} ORDER BY (SELECT NULL)) AS _rn
    FROM GOBGENER.dbo.EST_EXP_MIXTAS_M t1
    LEFT JOIN #cruce_ejec_exp_mixtas_m_dip t2
        ON t2.C_CUENTA = t1.C_CUENTA AND t2.C_ITEM = t1.C_ITEM AND t2.C_ASIGNACION = t1.C_ASIGNACION
       AND t2.C_ANALITICO = t1.C_ANALITICO AND t2.MES = t1.MES AND t2.ENTIDAD = t1.ENTIDAD
    LEFT JOIN #prom_mes t3
        ON t2.MONEDA = t3.MONEDA AND t2.MES = t3.MES AND t2.AÑO = t3.AÑO
    WHERE t2.DEVENG IS NOT NULL
)
SELECT * INTO #ejec_exp_mixtas_m_prelim_dip FROM base WHERE _rn = 1
"""
work_conn.execute(text(sql_ejec_exp_mixtas_m_prelim_dip))
work_conn.execute(text("ALTER TABLE #ejec_exp_mixtas_m_prelim_dip DROP COLUMN _rn"))

# EJEC_EXP_MIXTAS_M_DIP_I: igual que la preliminar, sin columnas de control
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_dip_i"))
res = work_conn.execute(text(
    "SELECT * INTO #ejec_exp_mixtas_m_dip_i FROM #ejec_exp_mixtas_m_prelim_dip"
))
work_conn.execute(text(
    "ALTER TABLE #ejec_exp_mixtas_m_dip_i DROP COLUMN DEVENG, DATO_OR, Clasif_Tcons, ENTIDAD"
))
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_exp_mixtas_m_prelim_dip"))
_log("CREATE #ejec_exp_mixtas_m_dip_i", res.rowcount)


In [ ]:
# Crea tabla con la ejecucion total de transacciones Mixtas (M), Muestra + Resto
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_mixtas_m_total_dip_imp"))
sql_ejec_mixtas_m_total_dip_imp = """
SELECT * INTO #ejec_mixtas_m_total_dip_imp
FROM (
    SELECT * FROM #ejec_muestra_mixtas_m_dip_i
    UNION ALL
    SELECT * FROM #ejec_exp_mixtas_m_dip_i
) u
"""
res = work_conn.execute(text(sql_ejec_mixtas_m_total_dip_imp))
_log("CREATE #ejec_mixtas_m_total_dip_imp", res.rowcount)


In [ ]:
# SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para CG coyuntura
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt_dip_imp"))
_trim_case = "CASE WHEN t1.MES IN (1, 2, 3) THEN 1 WHEN t1.MES IN (4, 5, 6) THEN 2 WHEN t1.MES IN (7, 8, 9) THEN 3 ELSE 4 END"
# El SELECT original repite C_CUP de t1 y de t2 (MOLDE_T_SCN_2): SAS conserva el
# ULTIMO (t2.C_CUP) cuando hay nombres duplicados, por eso t1.C_CUP solo queda en el GROUP BY
sql_ejec_g_mixt_dip_imp = f"""
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    {_trim_case} AS TRIM,
    t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_SCN, t1.C_CAGENTE,
    t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO,
    t2.C_CUP AS C_CUP,
    SUM(t1.DEVENGO) AS Dato, t1.ACTIVIDAD, t1.NIVEL
INTO #ejec_g_mixt_dip_imp
FROM #ejec_mixtas_m_total_dip_imp t1
LEFT JOIN GOBGENER.dbo.MOLDE_T_SCN_2 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
WHERE t1.C_SCN IN ('D11', 'P2', 'D63')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, {_trim_case}, t1.MES, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
         t1.C_SCN, t1.C_CAGENTE, t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL,
         t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP
"""
res = work_conn.execute(text(sql_ejec_g_mixt_dip_imp))
_log("CREATE #ejec_g_mixt_dip_imp", res.rowcount)


In [ ]:
# SALIDA de EJECUCION TOTAL DE TRANSACCIONES MIXTAS (M) para ANALISIS DESAGREGADO CG coyuntura
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_g_mixt_aper_dip_imp"))
_trim_case = "CASE WHEN t1.MES IN (1, 2, 3) THEN 1 WHEN t1.MES IN (4, 5, 6) THEN 2 WHEN t1.MES IN (7, 8, 9) THEN 3 ELSE 4 END"
sql_ejec_g_mixt_aper_dip_imp = f"""
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    {_trim_case} AS TRIM,
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM,
    t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
    t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO,
    t2.C_CUP AS C_CUP,
    SUM(t1.DEVENGO) AS Dato, t1.ACTIVIDAD, t1.NIVEL
INTO #ejec_g_mixt_aper_dip_imp
FROM #ejec_mixtas_m_total_dip_imp t1
LEFT JOIN GOBGENER.dbo.MOLDE_T_SCN_2 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
WHERE t1.C_SCN IN ('D11', 'P2', 'D63')
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, {_trim_case}, t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
         t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE,
         t1.C_CUP, t1.ACTIVIDAD, t1.NIVEL, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP
"""
res = work_conn.execute(text(sql_ejec_g_mixt_aper_dip_imp))
_log("CREATE #ejec_g_mixt_aper_dip_imp", res.rowcount)


In [ ]:
# Limpieza de tablas de trabajo intermedias; incluye el nombre con typo del SAS original
# (CRUCE_EJEC_EXP_MIXTAS_M_DIO) que nunca llego a existir, por eso es no-op
for _t in [
    "#ejec_mixtas_m_total_dip_imp",
    "#ejec_muestra_mixtas_m_dip_i",
    "#cruce_ejec_exp_mixtas_m_dio",
    "#ejec_exp_mixtas_m_dip_i",
    "#cruce_ejec_exp_mixtas_m_dip",
]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {_t}"))


## 3b_Imputa_Salida_Coy

Arma la salida final de ejecución presupuestaria coyuntural por transacción SCN: abre las transferencias en especie de salud pública con ratios externos, descarta analíticos e ingresos ya cubiertos por otras fuentes, incorpora imputaciones permanentes y la cobertura de ingresos operacionales agregados por DIPRES, y entrega el monto agrupado por periodo, cuenta y transacción

*confianza: medium · verificador: approve · SAS: PROC SQL: cadena de CREATE TABLE/DELETE/UPDATE/APPEND sobre datasets WORK, con JOINs a tablas de clasificación CR18_T_SCN_1/2 y a la tabla de imputaciones permanentes*

In [ ]:
# ========= 3b_Imputa_Salida_Coy =========
# /*Analiticos predefinidos */
cols_aper_prelim = (
    "SUBSECTOR, MONEDA, AÑO, [TRIM], MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, "
    "C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, "
    "C_CAGENTE, C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, DEVENGO"
)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper"))
sql_a_prelim = '''
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE WHEN t1.MES IN (1,2,3) THEN 1
         WHEN t1.MES IN (4,5,6) THEN 2
         WHEN t1.MES IN (7,8,9) THEN 3
         ELSE 4 END AS [TRIM],
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    CASE WHEN t1.ACTIVIDAD='FFAA' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='FFOO' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='PREVISION' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='FONASA' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='CORFO' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='CENABAST' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='AAPP' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='SSPP' THEN 'SSPP'
         WHEN t1.ACTIVIDAD='EEPP' THEN 'EEPP' END AS C_ACT,
    t1.NIVEL,
    -- FORMAT=COMMAX12.2 es solo formato de despliegue, no se traduce
    (t1.DEVENG * t3.DATO / 1000000) AS DEVENGO
INTO #ejecucion_a_prelim_dip_aper
FROM #ejec_dip_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA NOT IN ('05','13','24','33')
  AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.C_ENTIDAD NOT IN (5601,10201)
'''
work_conn.execute(text(sql_a_prelim))


In [ ]:
# /*nuevo, para eliminar ingresos operacionales que vienen agregados en data dipres. Se incorporan al final del proceso*/
work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_CUENTA='07' AND C_ITEM IN ('0','00')"))
work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper WHERE C_CUENTA='07' AND C_ASIGNACION NOT IN ('000')"))


In [ ]:
# /*Estructura para abrir transferencias en especie...*/
work_conn.execute(text("DROP TABLE IF EXISTS #ratios_2"))
sql_ratios_2 = '''
SELECT
    t1.SUBSECTOR, t1.AÑO, t1.MONEDA, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA,
    t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
    t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.Ratio
INTO #ratios_2
FROM #ratios t1
WHERE t1.C_SCN = 'D63'
'''
work_conn.execute(text(sql_ratios_2))


In [ ]:
# /*apertura transferencias en especie de la salud pública*/
# EJECUCION_A_prelim_DIP_aper se auto-referencia (lee su propia version anterior); se usa un
# nombre fisico nuevo (_v2) en vez de sobrescribir la misma tabla mientras se lee
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper_v2"))
sql_a_prelim_v2 = '''
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.[TRIM], t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
    t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO,
    t1.N_ANALITICO, t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA,
    t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL, t1.DEVENGO, t2.Ratio
INTO #ejecucion_a_prelim_dip_aper_v2
FROM #ejecucion_a_prelim_dip_aper t1
LEFT JOIN #ratios_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
   AND t1.C_CUENTA = t2.C_CUENTA AND t1.AÑO = t2.AÑO
'''
work_conn.execute(text(sql_a_prelim_v2))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_ajtesp_ap"))
sql_ajtesp = '''
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.[TRIM], t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
    t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    'D63' AS C_SCN, 'S14' AS C_CAGENTE, '433' AS C_TRANSACCION, 'transfesp' AS T_TRANSACCION, 'IG' AS N_CUENTA,
    t1.C_ACTIVO, '272' AS C_CUP, t1.C_ACT, t1.NIVEL,
    t1.DEVENGO * t1.Ratio AS DEVENGO
INTO #ejecucion_a_prelim_dip_ajtesp_ap
FROM #ejecucion_a_prelim_dip_aper_v2 t1
WHERE t1.Ratio IS NOT NULL
'''
work_conn.execute(text(sql_ajtesp))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_prelim_dip_aper_v3"))
sql_a_prelim_v3 = '''
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.[TRIM], t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
    t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
    t1.C_ACT, t1.NIVEL,
    CASE WHEN t1.Ratio IS NOT NULL THEN t1.DEVENGO * (1 - t1.Ratio) ELSE t1.DEVENGO END AS DEVENGO
INTO #ejecucion_a_prelim_dip_aper_v3
FROM #ejecucion_a_prelim_dip_aper_v2 t1
'''
work_conn.execute(text(sql_a_prelim_v3))


In [ ]:
# PROC DATASETS; APPEND BASE=EJECUCION_A_prelim_DIP_aper DATA=EJECUCION_A_prelim_DIP_ajtesp_ap FORCE
# /*fin apertura de transf en especie de salud pública*/
sql_append_ajtesp = f'''
INSERT INTO #ejecucion_a_prelim_dip_aper_v3 ({cols_aper_prelim})
SELECT {cols_aper_prelim}
FROM #ejecucion_a_prelim_dip_ajtesp_ap
'''
res = work_conn.execute(text(sql_append_ajtesp))
_log("APPEND #ejecucion_a_prelim_dip_aper_v3 (ajtesp)", res.rowcount)


In [ ]:
work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper_v3 WHERE C_ENTIDAD IN (10201)"))
work_conn.execute(text("DELETE FROM #ejecucion_a_prelim_dip_aper_v3 WHERE C_CUENTA='32' AND C_ANALITICO IN ('04001000','04002000')"))


In [ ]:
# /*Analiticos predefinidos: AJUSTE COYUNTURA - se elimina de indicador transacción CORFO
# que se desagrega anualmente entre ventas y D45 */
res = work_conn.execute(text(
    "DELETE FROM #ejecucion_a_prelim_dip_aper_v3 "
    "WHERE C_ENTIDAD = 3801 AND C_CUENTA='06' AND C_ITEM='01' "
    "AND C_ASIGNACION IN ('00','000') AND C_ANALITICO IN ('01000000')"
))
_log("DELETE #ejecucion_a_prelim_dip_aper_v3 (ajuste CORFO)", res.rowcount)


In [ ]:
# /*Analiticos predefinidos (sigue). BASE DESGREGADA*/
# El SELECT original agrupa por todas las columnas salvo DEVENGO, sin agregarla (sin SUM):
# PROC SQL toma el valor del primer registro del grupo; se replica con ROW_NUMBER
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_aper"))
sql_a_dip_aper = '''
WITH ranked AS (
    SELECT
        SUBSECTOR, MONEDA, AÑO, [TRIM], MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA,
        C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_TRANSACCION,
        T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, DEVENGO,
        ROW_NUMBER() OVER (
            PARTITION BY SUBSECTOR, MONEDA, AÑO, [TRIM], MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA,
                C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE,
                C_TRANSACCION, T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL
            ORDER BY (SELECT NULL)
        ) AS rn
    FROM #ejecucion_a_prelim_dip_aper_v3
)
SELECT
    SUBSECTOR, MONEDA, AÑO, [TRIM], MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA,
    C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_TRANSACCION,
    T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, DEVENGO AS Dato
INTO #ejecucion_a_dip_aper
FROM ranked
WHERE rn = 1
'''
work_conn.execute(text(sql_a_dip_aper))


In [ ]:
# /*Analiticos variables */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_b_dip_aper"))
sql_b = '''
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS [TRIM],
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    CASE WHEN t1.ACTIVIDAD='FFAA' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='FFOO' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='PREVISION' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='FONASA' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='CORFO' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='CENABAST' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='AAPP' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='SSPP' THEN 'SSPP'
         WHEN t1.ACTIVIDAD='EEPP' THEN 'EEPP' END AS C_ACT,
    t1.NIVEL,
    (t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_b_dip_aper
FROM #ejec_dip_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
   AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
   AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA = '32'
  AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t1.C_ANALITICO IN ('04001000','04002000')
  AND t1.C_ENTIDAD NOT IN (5601,10201)
'''
work_conn.execute(text(sql_b))
work_conn.execute(text("DELETE FROM #ejecucion_b_dip_aper WHERE C_ENTIDAD IN (10201)"))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_c_dip_aper"))
sql_c = '''
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS [TRIM],
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    CASE WHEN t1.ACTIVIDAD='FFAA' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='FFOO' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='PREVISION' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='FONASA' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='CORFO' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='CENABAST' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='AAPP' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='SSPP' THEN 'SSPP'
         WHEN t1.ACTIVIDAD='EEPP' THEN 'EEPP' END AS C_ACT,
    t1.NIVEL,
    (t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_c_dip_aper
FROM #ejec_dip_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
   AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
   AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA IN ('05','13','24','33')
  AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.OBS NOT IN ('CR18_difcoy','CR18_difact')
  AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND t2.C_SCN NOT IN ('TC')
'''
work_conn.execute(text(sql_c))
work_conn.execute(text("DELETE FROM #ejecucion_c_dip_aper WHERE C_ENTIDAD IN (10201)"))


In [ ]:
# /* Transacciones diferenciadas para coyuntura */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_d_dip_aper"))
sql_d = '''
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS [TRIM],
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_ACTIVO AS C_SCN,
    t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    CASE WHEN t1.ACTIVIDAD='FFAA' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='FFOO' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='PREVISION' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='FONASA' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='CORFO' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='CENABAST' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='AAPP' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='SSPP' THEN 'SSPP'
         WHEN t1.ACTIVIDAD='EEPP' THEN 'EEPP' END AS C_ACT,
    t1.NIVEL,
    (t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_d_dip_aper
FROM #ejec_dip_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
   AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
   AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS = 'CR18_difcoy'
  AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND t2.C_ACTIVO IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
  AND t2.C_SCN NOT IN ('TC')
'''
work_conn.execute(text(sql_d))


In [ ]:
# /* Transacciones diferenciadas por actividad */
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_f_dip_aper"))
sql_f = '''
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS [TRIM],
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    t2.C_CUP AS C_ACT,
    t1.NIVEL,
    (t1.DEVENG * t3.DATO / 1000000) AS Dato
INTO #ejecucion_f_dip_aper
FROM #ejec_dip_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_2 t2
    ON t1.C_PARTIDA = t2.C_PARTIDA AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA
   AND t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ITEM = t2.C_ITEM
   AND t1.C_ASIGNACION = t2.C_ASIGNACION AND t1.C_ANALITICO = t2.C_ANALITICO AND t1.C_ENTIDAD = t2.ENTIDAD
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t2.OBS = 'CR18_difact'
  AND t1.C_ENTIDAD NOT IN (5601,10201)
  AND t2.C_SCN NOT IN ('TC')
  AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
'''
work_conn.execute(text(sql_f))


In [ ]:
# /*FORMATEA TABLA DE TCONS, PARA UNIRLA A LAS OTRAS*/
# DATA step (C_ACTIVO='TC'; C_ACT=ACTIVIDAD) + UPDATE condicional + subsetting final,
# consolidados en un solo SELECT equivalente
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_g_dip_aper"))
sql_g = '''
SELECT
    SUBSECTOR, MONEDA, AÑO, [TRIM], MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA,
    C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_TRANSACCION,
    T_TRANSACCION, N_CUENTA,
    'TC' AS C_ACTIVO,
    C_CUP,
    CASE WHEN ACTIVIDAD IN ('FFAA','FFOO','PREVISION','FONASA','CORFO','CENABAST') THEN 'AAPP' ELSE ACTIVIDAD END AS C_ACT,
    NIVEL,
    Dato
INTO #ejecucion_g_dip_aper
FROM #ejec_g_mixt_aper_dip_imp
WHERE C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
'''
work_conn.execute(text(sql_g))


In [ ]:
# /*UNE TABLAS de Salida total de ejecución CG coy de data DIPRES*/
cols_cg = (
    "SUBSECTOR, MONEDA, AÑO, [TRIM], MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, C_TIPO_CUENTA, "
    "C_CUENTA, C_ITEM, C_ASIGNACION, C_ANALITICO, N_ANALITICO, C_SCN, C_CAGENTE, C_TRANSACCION, "
    "T_TRANSACCION, N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, Dato"
)
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper_prev"))
sql_cg_prev = f'''
SELECT {cols_cg} INTO #ejecucion_cg_dip_aper_prev FROM #ejecucion_a_dip_aper
UNION ALL SELECT {cols_cg} FROM #ejecucion_b_dip_aper
UNION ALL SELECT {cols_cg} FROM #ejecucion_c_dip_aper
UNION ALL SELECT {cols_cg} FROM #ejecucion_d_dip_aper
UNION ALL SELECT {cols_cg} FROM #ejecucion_g_dip_aper
UNION ALL SELECT {cols_cg} FROM #ejecucion_f_dip_aper
'''
work_conn.execute(text(sql_cg_prev))
# PROC SQL DROP TABLE de las tablas intermedias ya consolidadas en el UNION
for t in [
    "#ejecucion_a_prelim_dip_aper", "#ejecucion_a_prelim_dip_aper_v2", "#ejecucion_a_prelim_dip_aper_v3",
    "#ejecucion_a_dip_aper", "#ejecucion_b_dip_aper", "#ejecucion_c_dip_aper", "#ejecucion_d_dip_aper",
    "#ejecucion_g_dip_aper", "#ejecucion_f_dip_aper",
]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
res = work_conn.execute(text("UPDATE #ejecucion_cg_dip_aper_prev SET N_CUENTA = 'DIPRES'"))
_log("UPDATE #ejecucion_cg_dip_aper_prev", res.rowcount)


In [ ]:
# /*Incorporacion de imputaciones permanentes*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejec_cg_dip_aper_imput_perm"))
sql_imput_perm = f'''
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.[TRIM], t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
    t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION, t1.N_CUENTA, t1.C_ACTIVO, t1.C_CUP,
    CASE WHEN t1.C_ACT='FFAA' THEN 'AAPP'
         WHEN t1.C_ACT='FFOO' THEN 'AAPP'
         WHEN t1.C_ACT='PREVISION' THEN 'AAPP'
         WHEN t1.C_ACT='FONASA' THEN 'AAPP'
         WHEN t1.C_ACT='CORFO' THEN 'AAPP'
         WHEN t1.C_ACT='CENABAST' THEN 'AAPP'
         WHEN t1.C_ACT='AAPP' THEN 'AAPP'
         WHEN t1.C_ACT='SSPP' THEN 'SSPP'
         WHEN t1.C_ACT='EEPP' THEN 'EEPP' END AS C_ACT,
    t1.NIVEL,
    t1.Dato
INTO #ejec_cg_dip_aper_imput_perm
FROM GOBGENER.dbo.IMPUT_GC_DIP_APER t1
WHERE t1.AÑO = {int(Year)}
  AND t1.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
'''
work_conn.execute(text(sql_imput_perm))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper_cob"))
sql_cg_cob = f'''
SELECT {cols_cg} INTO #ejecucion_cg_dip_aper_cob FROM #ejecucion_cg_dip_aper_prev
UNION ALL SELECT {cols_cg} FROM #ejec_cg_dip_aper_imput_perm
'''
work_conn.execute(text(sql_cg_cob))


In [ ]:
# /*NUEVO: IMPUTA INGRESOS OPERACIONALES PARA LAS ENTIDADES QUE LO INFORMAN AGREGADO EN DIPRES, ASÍ SE MEJORA EL INDICADOR DE VENTAS*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_a_dip_07_cob"))
sql_a_07_cob = '''
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO,
    CASE WHEN t1.MES IN (1,2,3) THEN 1 WHEN t1.MES IN (4,5,6) THEN 2 WHEN t1.MES IN (7,8,9) THEN 3 ELSE 4 END AS [TRIM],
    t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO, t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA,
    t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t2.C_SCN, t2.C_CAGENTE, t2.C_TRANSACCION, t2.T_TRANSACCION, t2.N_CUENTA, t2.C_ACTIVO, t2.C_CUP,
    CASE WHEN t1.ACTIVIDAD='FFAA' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='FFOO' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='PREVISION' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='FONASA' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='CORFO' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='CENABAST' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='AAPP' THEN 'AAPP'
         WHEN t1.ACTIVIDAD='SSPP' THEN 'SSPP'
         WHEN t1.ACTIVIDAD='EEPP' THEN 'EEPP' END AS C_ACT,
    t1.NIVEL,
    (t1.DEVENG * t3.DATO / 1000000) AS DEVENGO
INTO #ejecucion_a_dip_07_cob
FROM #ejec_dip_i t1
LEFT JOIN GOBGENER.dbo.CR18_T_SCN_1 t2
    ON t1.C_TIPO_CUENTA = t2.T_CUENTA AND t1.C_CUENTA = t2.C_CUENTA AND t1.C_ANALITICO = t2.C_ANALITICO
LEFT JOIN #prom_mes t3
    ON t1.MONEDA = t3.MONEDA AND t1.MES = t3.MES AND t1.AÑO = t3.AÑO
WHERE t1.C_CUENTA = '07'
  AND t2.C_SCN IN ('D1','D11','D12','P2','P3','P72','D63','P1','P11')
'''
work_conn.execute(text(sql_a_07_cob))


In [ ]:
# /*Verifica que entidades informan solo el agregado para filtrarlas*/
work_conn.execute(text("DROP TABLE IF EXISTS #cuenta_item_07"))
sql_cuenta_item_07 = '''
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.[TRIM], t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
    t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_CUP,
    COUNT(t1.DEVENGO) AS id
INTO #cuenta_item_07
FROM #ejecucion_a_dip_07_cob t1
GROUP BY t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.[TRIM], t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
         t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_CUP
'''
work_conn.execute(text(sql_cuenta_item_07))


In [ ]:
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_dip_07"))
sql_dip_07 = '''
SELECT
    t1.SUBSECTOR, t1.MONEDA, t1.AÑO, t1.[TRIM], t1.MES, t1.C_ENTIDAD, t1.C_PARTIDA, t1.C_CAPITULO,
    t1.C_PROGRAMA, t1.C_TIPO_CUENTA, t1.C_CUENTA, t1.C_ITEM, t1.C_ASIGNACION, t1.C_ANALITICO, t1.N_ANALITICO,
    t1.C_SCN, t1.C_CAGENTE, t1.C_TRANSACCION, t1.T_TRANSACCION,
    'DIPRES' AS N_CUENTA,
    t1.C_ACTIVO, t1.C_CUP, t1.C_ACT, t1.NIVEL,
    t1.DEVENGO AS Dato
INTO #ejecucion_dip_07
FROM #ejecucion_a_dip_07_cob t1
LEFT JOIN #cuenta_item_07 t2
    ON t1.MONEDA = t2.MONEDA AND t1.C_ENTIDAD = t2.C_ENTIDAD AND t1.C_PARTIDA = t2.C_PARTIDA
   AND t1.C_CAPITULO = t2.C_CAPITULO AND t1.C_PROGRAMA = t2.C_PROGRAMA AND t1.C_TIPO_CUENTA = t2.C_TIPO_CUENTA
   AND t1.C_CUENTA = t2.C_CUENTA AND t1.MES = t2.MES
WHERE t2.id = 1
'''
work_conn.execute(text(sql_dip_07))


In [ ]:
# /*une imputacion de ingresos operacionales*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_aper_imp"))
sql_cg_imp = f'''
SELECT {cols_cg}, N'Imputación cobertura' AS OBS
INTO #ejecucion_cg_dip_aper_imp
FROM #ejecucion_cg_dip_aper_cob
UNION ALL
SELECT {cols_cg}, N'Imputación cobertura' AS OBS
FROM #ejecucion_dip_07
'''
work_conn.execute(text(sql_cg_imp))
work_conn.execute(text("DELETE FROM #ejecucion_cg_dip_aper_imp WHERE DATO = 0"))
work_conn.execute(text("DELETE FROM #ejecucion_cg_dip_aper_imp WHERE NIVEL = 'EXCLUIDO'"))


In [ ]:
# /*GENERA SALIDA AGRUPADA IMPUTACIONES POR COBERTURA DESDE DIPRES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cg_dip_imp"))
sql_cg_dip_imp = '''
SELECT
    SUBSECTOR, MONEDA, AÑO, [TRIM], MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE, C_TRANSACCION,
    N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, SUM(Dato) AS DATO, OBS
INTO #ejecucion_cg_dip_imp
FROM #ejecucion_cg_dip_aper_imp
GROUP BY SUBSECTOR, MONEDA, AÑO, [TRIM], MES, C_TIPO_CUENTA, C_CUENTA, C_ITEM, C_SCN, C_CAGENTE, C_TRANSACCION,
         N_CUENTA, C_ACTIVO, C_CUP, C_ACT, NIVEL, OBS
'''
work_conn.execute(text(sql_cg_dip_imp))
res = work_conn.execute(text("DELETE FROM #ejecucion_cg_dip_imp WHERE DATO = 0"))
_log("DELETE #ejecucion_cg_dip_imp (DATO=0)", res.rowcount)


In [ ]:
# Limpieza final de tablas temporales de este nodo (espeja el PROC SQL DROP TABLE del SAS original).
# Algunos nombres del DROP original (EJECUCION_CG_DIP_aper, CR18_APERTURA_PACAPR, EJECUCION_A_DIP_07, EJEC_DIP)
# no corresponden a ninguna tabla creada en este script; DROP TABLE IF EXISTS los ignora sin error.
for t in [
    "#ejecucion_cg_dip_aper", "#ejec_cg_dip_aper_imput_perm", "#cr18_apertura_pacapr", "#cuenta_item_07",
    "#ejecucion_a_dip_07", "#ejecucion_a_prelim_dip_ajtesp_ap", "#ejec_dip", "#ejecucion_dip_07",
    "#ejecucion_cg_dip_aper_prev", "#ratios_2",
]:
    work_conn.execute(text(f"DROP TABLE IF EXISTS {t}"))


In [ ]:
ejecucion_cg_dip_imp = pd.read_sql(text("SELECT * FROM #ejecucion_cg_dip_imp"), work_conn)
_log("ejecucion_cg_dip_imp", ejecucion_cg_dip_imp)


## 4_Salida_Imputaciones

Consolida la ejecución imputada de CGR y DIPRES por cobertura, en versión desagregada y agregada, y resume la cantidad de registros imputados por entidad y periodo

*confianza: medium · verificador: approve · SAS: DATA step SET (UNION de datasets) + PROC SQL CREATE TABLE con GROUP BY sobre columna CALCULATED*

In [ ]:
# ========= 4_Salida_Imputaciones =========
# /*SALIDA IMPUTACIONES POR COBERTURA DESAGREGADAS CGR-DIPRES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cobertura_desag"))
sql_ejecucion_cobertura_desag = """
SELECT *
INTO #ejecucion_cobertura_desag
FROM #ejecucion_cg_cgr_aper_imp
UNION ALL
SELECT *
FROM #ejecucion_cg_dip_aper_imp
"""
res = work_conn.execute(text(sql_ejecucion_cobertura_desag))
_log("EJECUCION_COBERTURA_DESAG", res.rowcount)


In [ ]:
# /*detalla entidades imputadas y periodo*/
work_conn.execute(text("DROP TABLE IF EXISTS #ent_cob_imp"))
sql_ent_cob_imp = """
SELECT
    C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA, MONEDA, AÑO, MES,
    CASE WHEN N_CUENTA = 'DIPRES' THEN 'DIPRES' ELSE 'CGR' END AS FUENTE,
    COUNT(DATO) AS N
INTO #ent_cob_imp
FROM #ejecucion_cobertura_desag
GROUP BY
    MONEDA, AÑO, MES, C_ENTIDAD, C_PARTIDA, C_CAPITULO, C_PROGRAMA,
    CASE WHEN N_CUENTA = 'DIPRES' THEN 'DIPRES' ELSE 'CGR' END
ORDER BY C_ENTIDAD, MONEDA, AÑO, MES
"""
res = work_conn.execute(text(sql_ent_cob_imp))
_log("ENT_COB_IMP", res.rowcount)


In [ ]:
# /*SALIDA IMPUTACIONES POR COBERTURA AGREGADAS CGR-DIPRES*/
work_conn.execute(text("DROP TABLE IF EXISTS #ejecucion_cobertura_ag"))
sql_ejecucion_cobertura_ag = """
SELECT *
INTO #ejecucion_cobertura_ag
FROM #ejecucion_cg_cgr_imp
UNION ALL
SELECT *
FROM #ejecucion_cg_dip_imp
"""
res = work_conn.execute(text(sql_ejecucion_cobertura_ag))
_log("EJECUCION_COBERTURA_AG", res.rowcount)
